# DataCo Supply Chain Analysis

**Question:** Where is this business losing money to operational failure, and are those failures concentrated or systemic?

**Dataset:** DataCo Smart Supply Chain, 180,519 order line items, 53 fields.

**Approach:** SQL analysis via DuckDB and JupySQL, with Power BI for visualization.

### Note on grain

The dataset is one row per **order line item**, not one row per order. Order level
counts require `DISTINCT "Order Id"`. Row counts overstate order volume. This governs
every aggregation below.

## Setup

In [1]:
import pandas as pd

df = pd.read_csv("DataCoSupplyChainDataset.csv", encoding="latin-1")

In [2]:
import duckdb

conn = duckdb.connect()
%load_ext sql
%sql conn
%config SqlMagic.displaylimit = 60

### Shape and completeness

In [3]:
print("Rows, columns:", df.shape)
print("Distinct orders:", df['Order Id'].nunique())
print("Distinct customers:", df['Customer Id'].nunique())
print()
print("Null counts (non-zero only):")
print(df.isna().sum()[df.isna().sum() > 0])
print()
print("Departments:", df['Department Id'].nunique())
print("Categories:", df['Category Id'].nunique())
print()
# Delivery Status and Shipping Mode are order level attributes repeated on every line
print("Orders with more than one Delivery Status:",
      (df.groupby('Order Id')['Delivery Status'].nunique() > 1).sum())

Rows, columns: (180519, 53)
Distinct orders: 65752
Distinct customers: 20652

Null counts (non-zero only):
Customer Lname              8
Customer Zipcode            3
Order Zipcode          155679
Product Description    180519
dtype: int64

Departments: 11
Categories: 51

Orders with more than one Delivery Status: 0


Order Zipcode (155,679 nulls) and Product Description (180,519 nulls) are unusable
and are excluded from all analysis.

---

# Section 1: Delivery performance

The dataset records both a scheduled shipping window and the actual days taken, which
allows the delivery problem to be decomposed into a fulfillment question and a
scheduling question.

## 1.1 Overall delivery status

Establishes the baseline before breaking the number down.

In [4]:
%%sql
WITH orders AS (
    SELECT DISTINCT "Order Id", "Delivery Status" FROM df
)
SELECT "Delivery Status",
       COUNT(*) AS "Orders",
       COUNT(*) * 100.0 / SUM(COUNT(*)) OVER() AS "Share of orders %"
FROM orders
GROUP BY "Delivery Status"
ORDER BY "Share of orders %" DESC

Running query in 'DuckDBPyConnection'

Delivery Status,Orders,Share of orders %
Late delivery,36048,54.82418785740358
Advance shipping,15127,23.006144299793164
Shipping on time,11722,17.827594597882953
Shipping canceled,2855,4.342073244920306


**Finding:** 54.8% of orders (36,048 of 65,752) are delivered late. This is the headline
number the rest of the section explains.

## 1.2 Late delivery by market

If lateness were a regional logistics problem, rates would vary by market.

In [5]:
%%sql
WITH orders AS (
    SELECT DISTINCT "Order Id", "Market", "Delivery Status" FROM df
)
SELECT "Market",
       COUNT(*) AS "Orders",
       SUM(CASE WHEN "Delivery Status" = 'Late delivery' THEN 1 ELSE 0 END) AS "Late orders",
       SUM(CASE WHEN "Delivery Status" = 'Late delivery' THEN 1 ELSE 0 END) * 100.0
           / COUNT(*) AS "Late %"
FROM orders
GROUP BY "Market"
ORDER BY "Late %" DESC

Running query in 'DuckDBPyConnection'

Market,Orders,Late orders,Late %
Pacific Asia,17577,9720,55.29953917050691
Europe,18561,10199,54.94854803081731
USCA,8579,4704,54.83156545051871
LATAM,17181,9339,54.35655666142832
Africa,3854,2086,54.12558380902958


**Finding:** The late rate is flat across markets, from 54.1% (Africa) to 55.3%
(Pacific Asia), a spread of about one percentage point.

**Implication:** This is not a regional problem. Regional intervention would not move
the number, so the cause sits upstream of geography.

## 1.3 Late delivery by department

Same test applied to product division rather than geography.

In [6]:
%%sql
SELECT "Department Id",
       SUM(CASE WHEN "Delivery Status" = 'Advance shipping' THEN 1 ELSE 0 END) * 100.0
           / COUNT(*) AS "Advance %",
       SUM(CASE WHEN "Delivery Status" = 'Shipping on time' THEN 1 ELSE 0 END) * 100.0
           / COUNT(*) AS "On time %",
       SUM(CASE WHEN "Delivery Status" = 'Late delivery' THEN 1 ELSE 0 END) * 100.0
           / COUNT(*) AS "Late %",
       SUM(CASE WHEN "Delivery Status" = 'Shipping canceled' THEN 1 ELSE 0 END) * 100.0
           / COUNT(*) AS "Cancelled %"
FROM df
GROUP BY "Department Id"
ORDER BY "Late %" DESC

Running query in 'DuckDBPyConnection'

Department Id,Advance %,On time %,Late %,Cancelled %
11,18.495934959349594,18.495934959349594,58.94308943089431,4.065040650406504
8,21.48148148148148,18.51851851851852,56.54320987654321,3.45679012345679
12,22.92817679558011,16.85082872928177,55.80110497237569,4.419889502762431
2,23.07382008874546,17.144009681323116,55.54659136748689,4.235578862444534
6,23.63204625232294,16.549659302085484,55.49246334916374,4.325831096427834
10,23.48122866894198,16.791808873720136,55.01706484641638,4.709897610921502
5,23.196869355809753,17.65201685731487,54.780252859723056,4.370860927152318
7,23.03734613601352,17.95815198695802,54.77483136656646,4.229670510462003
4,22.81725784725907,18.1211478019511,54.74713253602188,4.31446181476795
3,23.18760757314974,17.72117039586919,54.7263339070568,4.364888123924269


**Finding:** Late rates are similarly flat across departments. The four large
departments all sit at 54.7% to 54.8%. The only departments above 56% are the smallest
ones (under 500 line items each), where a slightly higher rate is normal noise.
Neither geography nor product division explains the failure.

## 1.4 Scheduled versus actual shipping days by mode

This is the query that identifies the mechanism. Comparing the promise against the
delivery isolates whether the failure is in fulfillment speed or in the commitment.

In [7]:
%%sql
SELECT "Shipping Mode",
       AVG("Days for shipment (scheduled)") AS "Scheduled days",
       AVG("Days for shipping (real)") AS "Actual days",
       AVG("Days for shipping (real)") - AVG("Days for shipment (scheduled)") AS "Gap"
FROM df
GROUP BY "Shipping Mode"
ORDER BY "Gap" DESC

Running query in 'DuckDBPyConnection'

Shipping Mode,Scheduled days,Actual days,Gap
Second Class,2.0,3.9908280327124035,1.9908280327124035
First Class,1.0,2.0,1.0
Same Day,0.0,0.4782787306151792,0.4782787306151792
Standard Class,4.0,3.995907268542579,-0.0040927314574208395


**Finding:** Scheduled days are fixed per tier. Every order in a tier receives an
identical promise regardless of distance, market, or product.

| Shipping Mode | Scheduled | Actual (avg) | Gap |
|---|---|---|---|
| Second Class | 2 | 4.0 | +2.0 |
| First Class | 1 | 2.0 | +1.0 |
| Same Day | 0 | 0.5 | +0.5 |
| Standard Class | 4 | 4.0 | 0.0 |

The averages hide the shape of the data, so the next query looks at the full range.

## 1.5 Distribution of actual shipping days by mode

An average of 4 days could mean every order arrives in 4 days, or that orders are
spread widely around 4. This query checks which.

In [8]:
%%sql
SELECT "Shipping Mode",
       "Days for shipment (scheduled)" AS "Scheduled",
       "Days for shipping (real)" AS "Actual days",
       COUNT(*) AS "Line items"
FROM df
GROUP BY ALL
ORDER BY "Shipping Mode", "Actual days"

Running query in 'DuckDBPyConnection'

Shipping Mode,Scheduled,Actual days,Line items
First Class,1,2,27814
Same Day,0,0,5080
Same Day,0,1,4657
Second Class,2,2,7138
Second Class,2,3,7065
Second Class,2,4,6978
Second Class,2,5,7052
Second Class,2,6,6983
Standard Class,4,2,21666
Standard Class,4,3,21700


**Finding:** Actual transit time does not cluster at 4 days. Standard and Second Class
are each spread evenly from 2 to 6 days, with roughly 20% of orders on each day.
First Class takes exactly 2 days every time. Same Day takes 0 or 1 day, about half each.

Standard and Second Class have the same delivery distribution. The only difference is
the promise: Standard promises 4 days, Second Class promises 2. Choosing Second Class
buys no faster delivery in this data.

**Data quality note:** A perfectly even 2 to 6 day spread, identical in every region, is
not what real logistics data looks like. This dataset is very likely simulated, so the
findings describe the structure of the data rather than a real company's operations.

## 1.6 Late rate by shipping mode

Late rate within each mode, and each mode's share of all late orders.

In [9]:
%%sql
WITH orders AS (
    SELECT DISTINCT "Order Id", "Shipping Mode", "Delivery Status" FROM df
)
SELECT "Shipping Mode",
       COUNT(*) AS "Orders",
       SUM(CASE WHEN "Delivery Status" = 'Late delivery' THEN 1 ELSE 0 END) AS "Late orders",
       SUM(CASE WHEN "Delivery Status" = 'Late delivery' THEN 1 ELSE 0 END) * 100.0
           / COUNT(*) AS "Late %",
       SUM(CASE WHEN "Delivery Status" = 'Late delivery' THEN 1 ELSE 0 END) * 100.0
           / SUM(SUM(CASE WHEN "Delivery Status" = 'Late delivery' THEN 1 ELSE 0 END)) OVER()
           AS "Share of all late orders %"
FROM orders
GROUP BY "Shipping Mode"
ORDER BY "Late %" DESC

Running query in 'DuckDBPyConnection'

Shipping Mode,Orders,Late orders,Late %,Share of all late orders %
First Class,10079,9602,95.26738763766247,26.63670661340435
Second Class,12778,9803,76.71779621223979,27.19429649356414
Same Day,3571,1648,46.149537944553344,4.571682201509099
Standard Class,39324,14995,38.13192961041603,41.59731469152241


**Finding:** An order is late exactly when its actual days exceed its scheduled days, so
the late rate in each tier follows directly from its promise:

| Shipping Mode | Late % | Share of all late orders |
|---|---|---|
| First Class | 95.3% | 26.6% |
| Second Class | 76.7% | 27.2% |
| Same Day | 46.1% | 4.6% |
| Standard Class | 38.1% | 41.6% |

First Class is late on every order that ships, because it promises 1 day and always
takes 2. The remaining 5% are cancelled orders.

**Implication:** Most lateness comes from the promise, not from slow fulfillment.
Moving the First Class promise from 1 day to 2, which is what it already delivers,
would remove about a quarter of all late orders with no operational change. Standard
Class still produces the largest share of late orders, because 40% of its shipments
take 5 or 6 days against a 4 day promise.

## 1.7 Late rate over time

Checks whether the late rate is stable or trending.

In [10]:
%%sql
SELECT DATE_TRUNC('quarter', STRPTIME("order date (DateOrders)", '%m/%d/%Y %H:%M')) AS "Quarter",
       COUNT(DISTINCT "Order Id") AS "Orders",
       SUM(CASE WHEN "Delivery Status" = 'Late delivery' THEN 1 ELSE 0 END) * 100.0
           / COUNT(*) AS "Late % (line items)"
FROM df
GROUP BY ALL
ORDER BY "Quarter"

Running query in 'DuckDBPyConnection'

Quarter,Orders,Late % (line items)
2015-01-01 00:00:00,5153,54.56432881333939
2015-04-01 00:00:00,5211,54.357431004674396
2015-07-01 00:00:00,5231,55.938136456211815
2015-10-01 00:00:00,5309,54.58888609504652
2016-01-01 00:00:00,5186,55.18448868426172
2016-04-01 00:00:00,5159,55.90500226493238
2016-07-01 00:00:00,5254,54.054054054054056
2016-10-01 00:00:00,5260,55.15525603073628
2017-01-01 00:00:00,5141,54.51842275371687
2017-04-01 00:00:00,5178,53.58527131782946


**Finding:** The late rate stays between 53.5% and 56.5% in every quarter from 2015
through January 2018, where the data ends. The problem is constant, not trending.

---

# Section 2: Profitability

Two questions: what does the business earn, and what does operational failure cost it.

## 2.1 Foregone profit from cancelled orders

Framed as foregone rather than lost. Cancelled orders never generated revenue, so this
is profit that did not materialize rather than a write-off. Revenue is deliberately not
used here, since unearned revenue is not a loss.

In [11]:
%%sql
SELECT
    SUM(CASE WHEN "Delivery Status" = 'Shipping canceled'
        THEN "Benefit per order" ELSE 0 END) AS "Foregone profit",
    SUM("Benefit per order") AS "Total profit",
    SUM(CASE WHEN "Delivery Status" = 'Shipping canceled'
        THEN "Benefit per order" ELSE 0 END) * 100.0
        / SUM("Benefit per order") AS "Foregone as % of total"
FROM df

Running query in 'DuckDBPyConnection'

Foregone profit,Total profit,Foregone as % of total
160482.34018122623,3966902.9740502946,4.045532276212197


In [12]:
%%sql
SELECT "Order Status",
       COUNT(DISTINCT "Order Id") AS "Orders",
       SUM("Benefit per order") AS "Foregone profit"
FROM df
WHERE "Delivery Status" = 'Shipping canceled'
GROUP BY "Order Status"

Running query in 'DuckDBPyConnection'

Order Status,Orders,Foregone profit
CANCELED,1367,75345.63014485993
SUSPECTED_FRAUD,1488,85136.71003636598


**Finding:** Orders that never shipped carry $160,482 of profit against $3,966,903 in
total, about 4%. They are 2,855 orders (4.3% of all orders) and split into two groups:

| Order Status | Orders | Foregone profit |
|---|---|---|
| Canceled | 1,367 | $75,346 |
| Suspected fraud | 1,488 | $85,137 |

**Implication:** This is the only failure in the dataset with a direct dollar figure.
Only the cancelled half is a clear recovery target. The suspected fraud half may be the
fraud screen working as intended, so the realistic target is closer to $75K.

## 2.2 Profit margin by department

Margin is computed as total profit divided by total revenue. Averaging the per order
profit ratio was rejected because that average weights every order equally regardless of
size, biasing the result toward small orders.

In [13]:
%%sql
SELECT "Department Id",
       SUM("Order Item Total") AS "Revenue",
       SUM("Order Profit Per Order") AS "Profit",
       SUM("Order Profit Per Order") * 100.0
           / SUM("Order Item Total") AS "Margin %"
FROM df
GROUP BY "Department Id"
ORDER BY "Margin %" DESC

Running query in 'DuckDBPyConnection'

Department Id,Revenue,Profit,Margin %
2,357158.2619040124,46538.06013663792,13.030094806863293
6,1125953.4392677564,145251.46005859392,12.900307862912662
4,7167654.214672526,881882.929553231,12.303647792439191
10,933829.2388320012,113170.01004233994,12.118919106012227
5,4142100.9573977645,497523.5598548718,12.011381783592167
7,15378581.710068671,1834155.435832589,11.926687846849557
9,205675.89941936548,24193.12000745697,11.762739375763273
3,3599468.828489284,410222.4986239577,11.396750969951592
12,95358.1102879,9493.629974264,9.955765635037599
11,37318.29984653005,3589.2599586140004,9.617962161659781


**Finding:** Overall margin is 12.0%. By department it ranges from 7.8% (Book Shop) to
13.0% (Fitness). The low end is occupied by departments with negligible revenue (Book
Shop sells $11K in total), so the volume weighted picture is tighter than the raw range
suggests. The four large departments all sit between 11.4% and 12.3%. No department is
structurally unprofitable and none is a standout.

## 2.3 Profit margin by category

In [14]:
%%sql
SELECT "Category Name",
       SUM("Order Item Total") AS "Revenue",
       SUM("Order Profit Per Order") AS "Profit",
       SUM("Order Item Discount") AS "Discount given",
       SUM("Order Profit Per Order") * 100.0
           / SUM("Order Item Total") AS "Margin %"
FROM df
GROUP BY "Category Name"
ORDER BY "Margin %" DESC

Running query in 'DuckDBPyConnection'

Category Name,Revenue,Profit,Discount given,Margin %
Golf Bags & Carts,9403.7900006,1810.069992479,965.6000041899998,19.248302996595097
Fitness Accessories,31751.360046089983,5258.390016191997,3850.159998737998,16.561148903728743
Toys,5485.679967798994,900.7100003560001,619.2699990499992,16.419295431800222
Soccer,23909.94973655001,3901.9499534469996,2567.120006182,16.31935657096875
Women's Clothing,126006.9303264001,19102.849930084,14276.439729002008,15.160158160032331
Baseball & Softball,84367.27085248992,12762.130067433005,9689.990069986006,15.126873180177498
Garden,231765.4602915999,33443.01010614101,26003.51957814801,14.429678203155932
Tennis & Racquet,40077.810346459984,5747.979996384998,4507.390045555999,14.342050992046548
Music,101687.2898662001,14436.319922849998,11435.050104522,14.196779107639975
CDs,2750.0299979349993,383.849999637,309.5599985469999,13.958029546049804


---

# Section 3: Commercial policy

Whether the business differentiates its pricing and discounting by customer value.

## 3.1 Share of orders receiving a discount

In [15]:
%%sql
SELECT "Department Id",
       COUNT(CASE WHEN "Order Item Discount Rate" != 0 THEN 1 END) * 100.0
           / COUNT("Order Id") AS "Discounted orders %"
FROM df
GROUP BY "Department Id"
ORDER BY "Discounted orders %" DESC

Running query in 'DuckDBPyConnection'

Department Id,Discounted orders %
2,94.83662767244857
8,94.5679012345679
11,94.51219512195122
12,94.47513812154696
7,94.47360942851587
5,94.45514750150512
4,94.44467121106985
9,94.42250740375124
10,94.40273037542661
3,94.35456110154905


**Finding:** 94.4% of line items carry a discount, and that share is 94% to 95% in
every department.

## 3.2 Discount rate versus total spend

If discounting were a lever, larger customers would receive better terms. The effective
discount rate is total discount divided by gross sales (price before discount).

In [16]:
%%sql
WITH customers AS (
    SELECT "Customer Id",
           SUM("Order Item Total") AS spent,
           SUM("Order Item Discount") * 100.0 / SUM("Sales") AS discount_rate
    FROM df
    GROUP BY "Customer Id"
), quintiles AS (
    SELECT NTILE(5) OVER (ORDER BY spent) AS quintile, spent, discount_rate
    FROM customers
)
SELECT quintile AS "Spend quintile (5 = biggest)",
       COUNT(*) AS "Customers",
       AVG(spent) AS "Avg spend",
       AVG(discount_rate) AS "Avg discount rate %",
       (SELECT CORR(spent, discount_rate) FROM customers) AS "Correlation, spend vs rate"
FROM quintiles
GROUP BY quintile
ORDER BY quintile

Running query in 'DuckDBPyConnection'

Spend quintile (5 = biggest),Customers,Avg spend,Avg discount rate %,"Correlation, spend vs rate"
1,4131,81.0063131235157,10.813795971975562,-0.02002190714360102
2,4131,337.2011843995738,9.658150046170197,-0.02002190714360102
3,4130,1247.2843571226517,10.087287621264933,-0.02002190714360102
4,4130,2341.890519134059,10.13164903882226,-0.02002190714360102
5,4130,3996.0036244572625,10.036911109560016,-0.02002190714360102


**Finding:** The average discount is about 10% of gross sales. Individual customers vary
(most fall between 2% and 18%), but the variation is unrelated to how much they spend.
Every spend quintile averages about 10%, and the correlation between spend and discount
rate is essentially zero (-0.02). The largest buyers receive no better terms than the
smallest.

**Implication:** Discounting functions as a blanket price reduction rather than a lever.
There is no volume incentive, no loyalty tier, and no negotiated pricing.

## 3.3 Customer segment mix

In [17]:
df['Customer Segment'].value_counts(normalize=True)

Customer Segment
Consumer       0.517973
Corporate      0.303508
Home Office    0.178519
Name: proportion, dtype: float64

---

# Section 4: Revenue structure

Where revenue concentrates, and what that implies about risk.

## 4.1 Customer concentration

In [18]:
%%sql
SELECT "Customer Id", "Customer Segment",
       SUM("Order Item Total") AS "Total spent",
       COUNT("Order Id") AS "Order count",
       SUM("Order Item Total") * 100.0
           / SUM(SUM("Order Item Total")) OVER() AS "% of revenue"
FROM df
GROUP BY "Customer Id", "Customer Segment"
ORDER BY "% of revenue" DESC
LIMIT 30

Running query in 'DuckDBPyConnection'

Customer Id,Customer Segment,Total spent,Order count,% of revenue
791,Corporate,9436.610088329999,43,0.028548723948426002
8766,Corporate,8400.98010666,38,0.02541561638303304
9371,Consumer,8222.670103,44,0.024876172342844557
2641,Consumer,8221.640129540001,43,0.024873056348042558
4249,Consumer,8198.150068539999,39,0.02480199149946414
1657,Consumer,8165.230022619999,42,0.024702397970162566
3710,Consumer,8112.1301271699995,42,0.024541754026765242
5654,Home Office,7918.750080189999,47,0.02395671834904946
5715,Corporate,7881.100091730001,44,0.023842815250675702
1288,Home Office,7869.02006526,42,0.023806269357844096


**Finding:** Revenue is spread across 20,652 customers. The single largest customer is
0.03% of revenue, and the top 30 combined are under 1%.

**Implication:** No key account risk. Losing any one customer would not be material.

## 4.2 Departmental concentration

Revenue and category breadth per department, to distinguish genuine concentration from
small departments that happen to carry one product line.

In [19]:
%%sql
SELECT "Department Id",
       COUNT(DISTINCT "Category Id") AS "Category count",
       SUM("Order Item Total") AS "Revenue",
       SUM("Order Item Total") * 100.0
           / SUM(SUM("Order Item Total")) OVER() AS "% of total revenue"
FROM df
GROUP BY "Department Id"
ORDER BY "Revenue" DESC

Running query in 'DuckDBPyConnection'

Department Id,Category count,Revenue,% of total revenue
7,6,15378581.710068671,46.525063539711745
4,7,7167654.214672526,21.684416291132276
5,3,4142100.9573977645,12.531162747255143
3,6,3599468.828489284,10.889529288974199
6,12,1125953.4392677564,3.4063645329784715
10,3,933829.2388320012,2.8251281874357907
2,7,357158.2619040124,1.0805164703802186
9,4,205675.89941936548,0.6222345122247694
12,1,95358.1102879,0.2884883810362494
11,1,37318.29984653005,0.11289963562875738


**Finding:** Four departments (Fan Shop, Apparel, Golf, Footwear) account for about $30.3M
of $33.1M in revenue, 91.6% of the total. The remaining seven are marginal, three of
them under $100K.

The single category departments are the smallest by revenue, so their apparent
concentration is a size artifact rather than a strategic choice. Reading concentration
percentages without checking the underlying volume would produce a misleading
conclusion.

## 4.3 Category concentration within departments

In [20]:
%%sql
WITH dept_cat AS (
    SELECT "Department Id", "Category Id",
           SUM("Order Item Total") AS "Category sales",
           SUM(SUM("Order Item Total")) OVER (PARTITION BY "Department Id")
               AS "Department sales"
    FROM df
    GROUP BY "Department Id", "Category Id"
)
SELECT "Department Id", "Category Id", "Category sales", "Department sales",
       "Category sales" * 100.0 / "Department sales" AS "Category % of department"
FROM dept_cat
WHERE "Category sales" * 100.0 / "Department sales" > 50
ORDER BY "Category % of department" DESC

Running query in 'DuckDBPyConnection'

Department Id,Category Id,Category sales,Department sales,Category % of department
11,72,37318.29984653005,37318.29984653005,100.0
12,69,95358.1102879,95358.1102879,100.0
8,59,11303.420032740001,11303.420032740001,99.99999999999999
3,9,3320250.6986390995,3599468.8284892794,92.2427962803787
5,24,2828708.5,4142100.9573978223,68.29163579289168
10,64,595395.0,933829.2388320001,63.75844482495521
4,17,3982856.5026169894,7167654.214672289,55.56708489737167


## 4.4 Revenue by order country

Top 20 destination countries by revenue.

In [21]:
%%sql
SELECT "Order Country",
       SUM("Order Item Total") AS "Revenue",
       COUNT("Order Id") AS "Order count",
       COUNT("Order Id") * 100.0 / SUM(COUNT("Order Id")) OVER()
           AS "% of all orders"
FROM df
GROUP BY "Order Country"
ORDER BY "Revenue" DESC
LIMIT 20

Running query in 'DuckDBPyConnection'

Order Country,Revenue,Order count,% of all orders
Estados Unidos,4385242.499329087,24840,13.760324397985807
Francia,2590323.4941207273,13222,7.324436762889225
México,2368426.0865444965,13172,7.296738847434342
Alemania,1862577.587930095,9564,5.298057268209995
Australia,1521410.4144700891,8497,4.706983752402794
Reino Unido,1450047.2328530313,7302,4.045003573031094
Brasil,1433100.9964266415,7987,4.424465014762989
China,1052752.5801145048,5758,3.189691943784311
Italia,963830.3387124853,4989,2.763698004088212
India,865968.8880714072,4783,2.6495825924140948


---

# Questions the data could not answer

Two lines of inquiry were opened and closed because the dataset does not support them.
Recording them prevents the same questions being asked twice, and prevents conclusions
being drawn from columns that carry no variation.

## Inventory availability

Product Status was intended to identify stockouts and slow moving inventory.

In [22]:
df['Product Status'].value_counts()

Product Status
0    180519
Name: count, dtype: int64

**Result:** Product Status is 0 for all 180,519 rows. No product is ever recorded as
unavailable, so no stockout, restocking, or inventory analysis is possible.

## Bulk versus retail pricing

Order Item Quantity was intended to test whether bulk purchases receive better terms.

In [23]:
print("Max order quantity:", df['Order Item Quantity'].max())
print()
print(df['Order Item Quantity'].value_counts().sort_index())

Max order quantity: 5

Order Item Quantity
1    99134
2    20315
3    20350
4    20335
5    20385
Name: count, dtype: int64


**Result:** Maximum order quantity is 5. There are no bulk orders in this dataset, so
bulk versus retail comparisons are not meaningful.

## Possible extensions

A classification model for late delivery would not add much here. Lateness is fully
determined by shipping mode and actual days, and actual days look randomly assigned.
A more useful extension would be running the same promise versus delivery check on a
real operational dataset.